# 모듈 (5/5): Agent Memory — 단기 기억과 장기 기억
## 지식(Memory & Knowledge) 시리즈 — LangGraph 메모리 인프라편

이 노트북은 AI 에이전트의 메모리 시스템을 **단계적으로 구현하고 직접 실행**합니다.

```
메모리 계층 구조
┌─────────────────────────────────────────────────────────┐
│  단기 기억 (Short-term Memory)                           │
│  ├─ 버퍼 메모리    : 모든 대화 유지                      │
│  ├─ 슬라이딩 윈도우: 최근 N개만 유지                     │
│  └─ 요약 메모리    : 오래된 대화 → LLM 요약 후 압축      │
├─────────────────────────────────────────────────────────┤
│  장기 기억 (Long-term Memory)                            │
│  ├─ MemorySaver   : 세션 간 기억 (메모리 내)             │
│  └─ SqliteSaver   : 프로세스 재시작 후에도 영구 저장     │
├─────────────────────────────────────────────────────────┤
│  크로스 스레드 메모리                                     │
│  └─ LangGraph Store: 사용자별 지식·선호도 영구 저장      │
├─────────────────────────────────────────────────────────┤
│  시맨틱 메모리 (Semantic Memory)                         │
│  └─ LLM 기반 관련 기억 검색                              │
└─────────────────────────────────────────────────────────┘
```

> 📦 **환경 설치·실행 명령**은 [`env_guides/M04_5_memory.md`](env_guides/M04_5_memory.md) 에 별도 정리되어 있습니다.
> 길고 반복되는 고급 메모리 클래스 구현은 [`agentic_lib/memory_advanced.py`](agentic_lib/memory_advanced.py) 로 분리해 두었습니다(노트북은 import 만).

### 이 시리즈 구성
1. `M04_1_embeddings.ipynb` — 임베딩 · 시맨틱 유사도 · 공급자 비교
2. `M04_2_vector_rag.ipynb` — 문서 전처리 · 벡터 DB 3종 · LangChain RAG
3. `M04_3_graph_rag.ipynb` — 지식 그래프 · Neo4j · LlamaIndex GraphRAG
4. `M04_4_agent_memory.ipynb` — 에이전트 메모리 **직접 구현** · Deep-Knowledge Agent
5. **`M04_5_memory.ipynb`** ← (현재) LangGraph **메모리 인프라**(체크포인터 · Store · 영속화)

> 🔁 **4편과의 관계** — [4편 `M04_4_agent_memory`](M04_4_agent_memory.ipynb) 에서는 메모리를 **직접 구현** 해
> 단기·에피소딕·시맨틱·작업 메모리의 구조를 이해하고 Hybrid RAG 와 결합했습니다.
> 이 노트북은 같은 개념을 **LangGraph/LangChain 이 제공하는 표준 부품**
> (체크포인터 `MemorySaver`/`SqliteSaver` · 크로스 스레드 `Store` · 요약 메모리)으로 다시 세웁니다.
> 4편이 "왜 그렇게 동작하는가"라면, 5편은 "실무에서 무엇을 쓰는가" 입니다.
> 4편에 있던 **RAG 결합·multi-hop 추론** 은 여기서 다루지 않고, 대신 **영속화·세션 분리·사용자 프로필** 을 다룹니다.


---
## 0. 환경 설정


In [3]:
import sys, os
sys.path.insert(0, os.path.abspath(''))  # notebooks/ 를 import 경로에 추가

import utils
utils.reload_env()  # .env 재로드 (LLM_PROVIDER 등 갱신) + 현재 공급자 상태 출력

from utils import uv_install, get_llm, test_llm_connection, LLM_PROVIDER

# 공통 라이브러리 — 부트스트랩 + 고급 메모리 패턴(구현은 라이브러리에 분리)
from agentic_lib import bootstrap, memory, memory_advanced
from agentic_lib.bootstrap import to_text  # 공급자 무관 응답 정규화(<think> 제거 포함)
from agentic_lib.memory_advanced import (
    BufferMemory, SlidingWindowMemory, SummaryMemory,
    SemanticMemoryStore, FullMemoryAgent, make_store_memory_tools,
    DEFAULT_SYSTEM as SYSTEM,   # 단기 메모리 클래스/그래프 노드가 공유하는 기본 시스템 메시지
)

# 이 노트북이 추가로 필요로 하는 패키지(자세한 설치는 env_guides 참고)
uv_install([
    'langgraph>=0.2.32',
    'langchain>=0.3.0',
    'langchain-core>=0.3.0',
    'langgraph-checkpoint-sqlite',
])


llm = test_llm_connection()
print(f'\n준비 완료: {LLM_PROVIDER} 공급자')

# SQLite DB 저장 디렉터리
os.makedirs('workspace', exist_ok=True)

LLM 공급자: openrouter
  OpenRouter Key: 설정됨  /  Model: nvidia/nemotron-3-super-120b-a12b:free
[uv] 설치 완료: ['langgraph>=0.2.32', 'langchain>=0.3.0', 'langchain-core>=0.3.0', 'langgraph-checkpoint-sqlite']
LLM 연결 성공 [openrouter]: 1+1을 계산하면 2입니다.

준비 완료: openrouter 공급자


---
## 1. 단기 기억 (Short-term Memory)

단기 기억은 **현재 대화 세션** 내에서만 유지되는 맥락 정보입니다.
LLM 은 기본적으로 상태가 없기(stateless) 때문에, 이전 메시지를 직접 전달해야 합니다.

| 방식 | 장점 | 단점 |
|------|------|------|
| **버퍼 메모리** | 모든 맥락 유지 | 대화가 길어지면 토큰 폭발 |
| **슬라이딩 윈도우** | 토큰 제한 예측 가능 | 오래된 정보 소실 |
| **요약 메모리** | 핵심 정보 압축 유지 | 요약 LLM 호출 추가 비용 |


### 1-1. 버퍼 메모리 — 모든 대화를 그대로 유지


In [4]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# BufferMemory 구현은 agentic_lib.memory_advanced 로 분리(여기서는 사용만).
# 시스템 메시지는 SYSTEM(= memory_advanced.DEFAULT_SYSTEM)을 클래스가 내부적으로 사용한다.
buffer = BufferMemory()


def chat_buffer(user_input: str) -> str:
    """버퍼 메모리에 한 턴을 기록하고 LLM 응답(정규화된 문자열)을 반환한다."""
    buffer.add(HumanMessage(content=user_input))
    reply = to_text(llm.invoke(buffer.get()).content)  # 공급자 무관 정규화
    buffer.add(AIMessage(content=reply))
    return reply


print('=' * 60)
print('버퍼 메모리 — 다중 턴 대화')
print('=' * 60)

conversations = [
    '안녕하세요! 저는 김민준이고 인천에 살아요.',
    '직업은 백엔드 개발자이고 Python 을 주로 씁니다.',
    '취미는 주말마다 등산하는 거예요.',
    '방금 제가 말한 내용을 요약해줄 수 있나요?',
]

for i, msg in enumerate(conversations, 1):
    reply = chat_buffer(msg)
    est = buffer.token_estimate()
    print(f'[{i}] 사용자: {msg}')
    print(f'    AI: {reply[:100]}')
    print(f'    (누적 ~{est} 토큰, 메시지 {len(buffer.messages)}개)')
    print()


버퍼 메모리 — 다중 턴 대화
[1] 사용자: 안녕하세요! 저는 김민준이고 인천에 살아요.
    AI: 안녕하세요, 민준 씨! 인천에 사시는군요. 반갑습니다. 오늘 어떻게 도와드릴까요? 😊
    (누적 ~17 토큰, 메시지 2개)

[2] 사용자: 직업은 백엔드 개발자이고 Python 을 주로 씁니다.
    AI: 안녕하세요, 민준 씨! 인천에서 백엔드 개발자로 활동하시는군요. Python을 주로 사용하신다니, 웹 프레임워크(Django, FastAPI 등)나 데이터 처리, API 개발 쪽에
    (누적 ~86 토큰, 메시지 4개)

[3] 사용자: 취미는 주말마다 등산하는 거예요.
    AI: 등산이 취미라니 정말 멋져요! 😄 개발 업무로 인해 장시간 앉아 계실 텐데, 주말에 자연 속에서 몸을 움직이시며 머리를 맑게 하시는 모습이 떠올라요. 인천 근처에서 특히 좋아하는 
    (누적 ~149 토큰, 메시지 6개)

[4] 사용자: 방금 제가 말한 내용을 요약해줄 수 있나요?
    AI: 물론입니다! 민준 씨가 이전에 공유해주신 내용을 요약해 드리면 다음과 같습니다:

- **이름**: 김민준  
- **거주지**: 인천  
- **직업**: 백엔드 개발자 (주로 
    (누적 ~200 토큰, 메시지 8개)



### 1-2. 슬라이딩 윈도우 메모리 — 최근 N개 메시지만 유지

`collections.deque(maxlen=N)` 으로 구현합니다.
새 메시지가 들어오면 가장 오래된 메시지가 자동으로 제거됩니다.


In [6]:
# SlidingWindowMemory 구현은 agentic_lib.memory_advanced 로 분리(여기서는 사용만).
# maxlen=4 → 사용자 2턴 + AI 2턴만 유지
window = SlidingWindowMemory(max_messages=4)


def chat_window(user_input: str) -> str:
    """슬라이딩 윈도우에 한 턴을 기록하고 LLM 응답을 반환한다."""
    window.add(HumanMessage(content=user_input))
    reply = to_text(llm.invoke(window.get()).content)
    window.add(AIMessage(content=reply))
    return reply


print('=' * 65)
print('슬라이딩 윈도우 (maxlen=4) — 오래된 메시지 자동 삭제 확인')
print('=' * 65)

window_conv = [
    '안녕! 나는 이수진이야. 서울에 살아.',       # 곧 window 밖으로 밀림
    '나는 디자이너야. UI/UX 가 전문이야.',       # 곧 window 밖으로 밀림
    '최근에 Figma 공부를 시작했어.',
    '내 이름이 뭐라고 했지?',                    # 첫 메시지 소실 → 기억 못할 수도
]

for i, msg in enumerate(window_conv, 1):
    reply = chat_window(msg)
    print(f'[{i}] 사용자: {msg}')
    print(f'    AI: {reply[:110]}')
    print(f'    (현재 버퍼: {len(window)}개 / 최대 4개)')
    print()

print('--- 버퍼에 남아있는 메시지 ---')
for m in window.messages:
    role = '사용자' if isinstance(m, HumanMessage) else 'AI'
    print(f'  [{role}] {m.content[:60]}')


슬라이딩 윈도우 (maxlen=4) — 오래된 메시지 자동 삭제 확인
[1] 사용자: 안녕! 나는 이수진이야. 서울에 살아.
    AI: 안녕하세요, 수진 씨! 서울에 사신다니 반가워요. 오늘은 어떻게 도와드릴까요? 궁금한 점이나 도움이 필요한 일이 있으면 편하게 말씀해주세요! 😊
    (현재 버퍼: 2개 / 최대 4개)

[2] 사용자: 나는 디자이너야. UI/UX 가 전문이야.
    AI: 와, UI/UX 디자이너라니 정말 멋져요! 사용자 경험을 설계하는 일은 기술과 인간의 니트를 연결하는 다리 같은 역할이라서 특히 중요하고 창의적인 분야죠. 서울에서 활동하신다니, 트렌디한 디자인 씬
    (현재 버퍼: 4개 / 최대 4개)

[3] 사용자: 최근에 Figma 공부를 시작했어.
    AI: 정말 멋진 선택이에요! Figma는 현재 UI/UX 디자인의 산업 표준으로 자리매김했고, 특히 협업과 프로토타이핑에서 강점이 커서 서울의 스타트업이나 대기업 디자인 팀에서도 активно 사용되고 
    (현재 버퍼: 4개 / 최대 4개)

[4] 사용자: 내 이름이 뭐라고 했지?
    AI: 아이고, 죄송해요! 제가 실수로 이름을 언급한 적이 없었네요 😅  
대화 속에서 이름을 공유해주신 부분이 없어서 아직 모르고 있어요.  
혹시 편하게 부르실 이름이 있다면 알려주세요! (예: "민수
    (현재 버퍼: 4개 / 최대 4개)

--- 버퍼에 남아있는 메시지 ---
  [사용자] 최근에 Figma 공부를 시작했어.
  [AI] 정말 멋진 선택이에요! Figma는 현재 UI/UX 디자인의 산업 표준으로 자리매김했고, 특히 협업과 프로토
  [사용자] 내 이름이 뭐라고 했지?
  [AI] 아이고, 죄송해요! 제가 실수로 이름을 언급한 적이 없었네요 😅  
대화 속에서 이름을 공유해주신 부분이 없


### 1-3. 요약 메모리 (Summary Memory) — 오래된 대화를 LLM 으로 압축

버퍼 메모리와 슬라이딩 윈도우의 장점을 결합합니다.
- 최근 N개 메시지는 **원본 그대로** 유지
- 그보다 오래된 메시지는 **LLM 으로 요약**하여 압축

```
[오래된 대화] → LLM 요약 → [요약문]
[최근 대화]   →            [최근 대화 그대로]
                          ↓
                LLM 에 전달: 요약문 + 최근 대화
```


In [7]:
# SummaryMemory 구현은 agentic_lib.memory_advanced 로 분리(여기서는 사용만).
summary_mem = SummaryMemory(llm=llm, max_recent=4)


def chat_summary(user_input: str) -> str:
    """요약 메모리에 한 턴을 기록하고 LLM 응답을 반환한다(오래된 대화는 자동 요약)."""
    summary_mem.add(HumanMessage(content=user_input))
    reply = to_text(llm.invoke(summary_mem.get_context()).content)
    summary_mem.add(AIMessage(content=reply))
    return reply


print('=' * 65)
print('요약 메모리 — 오래된 대화가 자동 압축되는 과정')
print('=' * 65)

long_conv = [
    '안녕! 저는 박민수이고 광주에 살아요.',
    '직업은 데이터 분석가이고 4년차예요.',
    'Python 과 SQL 을 주로 써요.',
    '최근 LangGraph 를 공부하기 시작했어요.',
    '취미는 사이클링이에요. 주말마다 한강을 달려요.',
    '좋아하는 음식은 스시예요.',
    '지금까지 제 소개를 요약해줄 수 있어요?',  # 요약문을 바탕으로 답해야 함
]

for i, msg in enumerate(long_conv, 1):
    reply = chat_summary(msg)
    print(f'[{i}] 사용자: {msg}')
    print(f'    AI: {reply[:130]}')
    print()


요약 메모리 — 오래된 대화가 자동 압축되는 과정
[1] 사용자: 안녕! 저는 박민수이고 광주에 살아요.
    AI: 안녕하세요, 민수님! 광주에 사시는군요. 😊 광주는 맛있는 음식(특히 김치와 비빔밥!)과 아름다운 무등산 풍경으로 유명한 곳이라 정말 살기 좋은 동네 같아요. 혹시 광주에서 추천할 만한 곳이나 도움이 필요한 것이 있으면 언제든 편하게

[2] 사용자: 직업은 데이터 분석가이고 4년차예요.
    AI: 와, 4년차 데이터 분석가라니 정말 멋져요! 📊 광주에서 데이터 분야에서 커리어를 쌓아가고 계신다니 특히 흥미롭네요. 광주에도 최근에 스마트시티 관련 프로젝트나 지역 빅데이터 센터 같은 이니셔티브들이 생기고 있어서, 로컬에서 의미 있

[3] 사용자: Python 과 SQL 을 주로 써요.
    AI: Okay, the<unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><

  [요약 갱신] - 이름: 박민수  
- 직업: 데이터 분석가 (4년차)  
- 거주지: 광주  
- 취미: 현재 언급된 취미 없음...
[4] 사용자: 최근 LangGraph 를 공부하기 시작했어요.
    AI: 와, LangGraph 공부를 시작하셨다니 정말 시의적절하고 흥미로운 선택이네요! 🚀 데이터 분석가로서 Python과 SQL을 다뤄오신 분이 LangGraph를 배우면 **기존 업무에 LLM 기반 자동화/지능화를 입히는 강력한 툴킷*

[5] 사용자: 취미는 사이클링이에요. 주말마다 한강을 달려요.
    AI: 와, 사이클링 취미 정말 멋져요! 🚴‍♂️ 주말마다 한강을 달리신다니 — 특히 광주에 계신 분이 한강을 찾으신다니, 주말에 서울까지 가시며 라이딩을 즐기시는 건가요? (혹은 광주에도 "한강"과 같은 이름의 강이 있나 궁금해졌어요! 실

  [요약 갱신] - 이름: 박민수  
- 직업: 데이

### 1-4. LangGraph MessagesState — 그래프 기반 단기 기억

LangGraph 의 `MessagesState` 는 메시지 리스트를 그래프 상태로 관리합니다.
**MemorySaver 체크포인터**와 결합하면 thread_id 별로 대화 기록이 자동 누적됩니다.

```python
# 그래프 실행 흐름
invoke({messages: [새 메시지]}, config={thread_id: ...})
    → MemorySaver 에서 이전 메시지 로드
    → 새 메시지 추가
    → LLM 응답 생성
    → 전체 메시지 MemorySaver 에 저장
```


In [8]:
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.checkpoint.memory import MemorySaver


def llm_node(state: MessagesState) -> dict:
    """LLM 호출 노드 — 시스템 메시지 + 전체 히스토리 전달, 응답은 to_text 로 정규화."""
    messages = [SYSTEM] + state['messages']
    response = llm.invoke(messages)
    # 그래프 상태에 깔끔한 문자열만 누적되도록 정규화한 AIMessage 로 반환
    return {'messages': [AIMessage(content=to_text(response.content))]}


# 그래프 구성
builder = StateGraph(MessagesState)
builder.add_node('llm', llm_node)
builder.add_edge(START, 'llm')

mem_saver = MemorySaver()
graph = builder.compile(checkpointer=mem_saver)


def chat_graph(thread_id: str, user_input: str) -> str:
    """thread_id 별로 대화 기록이 자동 누적되는 그래프를 호출한다."""
    config = {'configurable': {'thread_id': thread_id}}
    result = graph.invoke({'messages': [HumanMessage(content=user_input)]}, config=config)
    return result['messages'][-1].content


print('=' * 60)
print('LangGraph MessagesState + MemorySaver 대화')
print('=' * 60)

tid = 'user-langgraph-demo'

turns = [
    '안녕! 나는 이준혁이야. 대전에 살고 있어.',
    '나는 ML 엔지니어야. 주로 PyTorch 를 써.',
    '내 이름이 뭐고 어디 산다고 했지?',
]

for i, msg in enumerate(turns, 1):
    reply = chat_graph(tid, msg)
    print(f'[{i}] 사용자: {msg}')
    print(f'    AI: {reply[:120]}')

# 저장된 상태 확인
config = {'configurable': {'thread_id': tid}}
state = graph.get_state(config)
print(f'\n저장된 메시지 수: {len(state.values["messages"])}개')
for m in state.values['messages']:
    role = '사용자' if isinstance(m, HumanMessage) else 'AI'
    print(f'  [{role}] {m.content[:60]}')


LangGraph MessagesState + MemorySaver 대화
[1] 사용자: 안녕! 나는 이준혁이야. 대전에 살고 있어.
    AI: 안녕, 이준혁! 대전에 사신다니 반가워요. 대전이 과학과 기술의 중심지로 유명해서 혹시 관심 있는 분야가 있나요? 오늘 어떻게 도와드릴까요? 😊
[2] 사용자: 나는 ML 엔지니어야. 주로 PyTorch 를 써.
    AI: 안녕하세요, 이준혁 님! 대전에서 ML 엔지니어로 활동하시다니 정말 멋지네요. 대전이 KAIST, ETRI 등 연구 기관이 밀집해 있어 ML 분야에서 활발한 교류가 이루어지는 곳이라서 특히 반갑습니다. 😊  

Py
[3] 사용자: 내 이름이 뭐고 어디 산다고 했지?
    AI: 당신의 이름은 **이준혁**이고, **대전에 산다고** 말씀하셨습니다! 😊  
혹시 다른 부분도 다시 확인하고 싶으신가요? 언제든 편하게 말씀해주세요!

저장된 메시지 수: 6개
  [사용자] 안녕! 나는 이준혁이야. 대전에 살고 있어.
  [AI] 안녕, 이준혁! 대전에 사신다니 반가워요. 대전이 과학과 기술의 중심지로 유명해서 혹시 관심 있는 분야가 있
  [사용자] 나는 ML 엔지니어야. 주로 PyTorch 를 써.
  [AI] 안녕하세요, 이준혁 님! 대전에서 ML 엔지니어로 활동하시다니 정말 멋지네요. 대전이 KAIST, ETRI 
  [사용자] 내 이름이 뭐고 어디 산다고 했지?
  [AI] 당신의 이름은 **이준혁**이고, **대전에 산다고** 말씀하셨습니다! 😊  
혹시 다른 부분도 다시 확인하


---
## 2. 장기 기억 (Long-term Memory)

장기 기억은 **세션을 넘어서도 유지**되는 기억입니다.
LangGraph 체크포인터가 그래프의 전체 상태를 저장하여 재시작 후에도 이어서 대화할 수 있습니다.

| 체크포인터 | 저장 위치 | 특징 |
|-----------|----------|------|
| `MemorySaver` | 프로세스 메모리 | 빠름, 재시작 시 소실 |
| `SqliteSaver` | SQLite 파일 | 재시작 후에도 유지, 파일 크기 증가 |
| `PostgresSaver` | PostgreSQL DB | 프로덕션용, 분산 환경 지원 |


### 2-1. MemorySaver 심화 — 멀티 스레드 & 상태 검사


In [9]:
# MemorySaver 는 이미 위에서 생성한 graph + mem_saver 를 재사용합니다

print('=' * 65)
print('멀티 스레드: 사용자별 완전히 독립된 기억')
print('=' * 65)

# 사용자 A: 앨리스
print('\n[사용자 A: alice]')
chat_graph('alice', '나는 앨리스야. 뉴욕에서 마케터로 일하고 있어.')
chat_graph('alice', '영어와 스페인어를 할 줄 알아.')
r = chat_graph('alice', '내 직업이 뭐라고 했지?')
print(f'앨리스의 직업 질문 답변: {r[:100]}')

# 사용자 B: 밥 (앨리스 정보 없음)
print('\n[사용자 B: bob]')
chat_graph('bob', '나는 밥이야. 도쿄에서 엔지니어로 일해.')
r = chat_graph('bob', '앨리스가 어디 산다고 했어?')  # 앨리스 정보 없음
print(f'밥이 앨리스 정보 물을 때: {r[:100]}')

# 각 스레드의 메시지 수 확인
print('\n[각 스레드 상태]')
for tid in ['alice', 'bob', 'user-langgraph-demo']:
    cfg = {'configurable': {'thread_id': tid}}
    try:
        st = graph.get_state(cfg)
        n = len(st.values.get('messages', []))
        print(f'  {tid}: {n}개 메시지')
    except Exception:
        print(f'  {tid}: 기록 없음')


멀티 스레드: 사용자별 완전히 독립된 기억

[사용자 A: alice]
앨리스의 직업 질문 답변: Okay, the user just asked, "내 직업이 뭐라고 했지?" which means "What did you say my job was?" in Korean. Let

[사용자 B: bob]
밥이 앨리스 정보 물을 때: I don’t recall any mention of **Alice** in our conversation so far — you introduced yourself as **Bo

[각 스레드 상태]
  alice: 6개 메시지
  bob: 4개 메시지
  user-langgraph-demo: 6개 메시지


### 2-2. 대화 히스토리 타임라인 조회

`get_state_history()` 로 체크포인트 전체 이력을 조회할 수 있습니다.
각 체크포인트는 **그래프 실행 직후의 스냅샷**입니다.


In [10]:
config = {'configurable': {'thread_id': 'alice'}}

print('=' * 60)
print('앨리스 스레드 — 체크포인트 히스토리')
print('=' * 60)

history = list(graph.get_state_history(config))
print(f'총 체크포인트 수: {len(history)}\n')

for i, checkpoint in enumerate(reversed(history)):
    msgs = checkpoint.values.get('messages', [])
    last_msg = msgs[-1].content[:50] if msgs else '(없음)'
    ts = checkpoint.metadata.get('created_at', 'N/A') if checkpoint.metadata else 'N/A'
    print(f'[체크포인트 {i+1}] 메시지 {len(msgs)}개 | 마지막: {last_msg}')


앨리스 스레드 — 체크포인트 히스토리
총 체크포인트 수: 9

[체크포인트 1] 메시지 0개 | 마지막: (없음)
[체크포인트 2] 메시지 1개 | 마지막: 나는 앨리스야. 뉴욕에서 마케터로 일하고 있어.
[체크포인트 3] 메시지 2개 | 마지막: Nice to meet you, Alice! 👋 Marketing in NYC must b
[체크포인트 4] 메시지 2개 | 마지막: Nice to meet you, Alice! 👋 Marketing in NYC must b
[체크포인트 5] 메시지 3개 | 마지막: 영어와 스페인어를 할 줄 알아.
[체크포인트 6] 메시지 4개 | 마지막: That's fantastic, Alice! 🌎 Being bilingual in Engl
[체크포인트 7] 메시지 4개 | 마지막: That's fantastic, Alice! 🌎 Being bilingual in Engl
[체크포인트 8] 메시지 5개 | 마지막: 내 직업이 뭐라고 했지?
[체크포인트 9] 메시지 6개 | 마지막: Okay, the user just asked, "내 직업이 뭐라고 했지?" which m


### 2-3. SqliteSaver — 프로세스 재시작 후에도 기억 유지


In [11]:
try:
    import sqlite3
    from langgraph.checkpoint.sqlite import SqliteSaver

    DB_PATH = os.path.join('workspace', 'agent_memory.db')
    conn = sqlite3.connect(DB_PATH, check_same_thread=False)
    sqlite_saver = SqliteSaver(conn)

    # SqliteSaver 로 그래프 재컴파일
    builder2 = StateGraph(MessagesState)
    builder2.add_node('llm', llm_node)
    builder2.add_edge(START, 'llm')
    persistent_graph = builder2.compile(checkpointer=sqlite_saver)

    def chat_persistent(thread_id: str, user_input: str) -> str:
        config = {'configurable': {'thread_id': thread_id}}
        result = persistent_graph.invoke(
            {'messages': [HumanMessage(content=user_input)]}, config=config
        )
        return result['messages'][-1].content

    print('=' * 65)
    print(f'SqliteSaver — 대화를 {DB_PATH} 에 영구 저장')
    print('=' * 65)

    # 1번 대화 (커널 재시작 후에도 이 기록이 남음)
    r1 = chat_persistent('persistent-user-1', '저는 한지민이고 제주도에 살아요.')
    print(f'[1] AI: {r1[:100]}')

    r2 = chat_persistent('persistent-user-1', '저는 관광업에 종사하고 있어요.')
    print(f'[2] AI: {r2[:100]}')

    r3 = chat_persistent('persistent-user-1', '제가 어디 살고 무슨 일 한다고 했죠?')
    print(f'[3] AI: {r3[:120]}')

    # DB 파일 크기 확인
    db_size = os.path.getsize(DB_PATH)
    print(f'\nDB 파일 크기: {db_size:,} bytes ({DB_PATH})')
    print('→ 이 DB 파일을 유지하면 커널/프로세스 재시작 후에도 기억이 유지됩니다.')

except ImportError as e:
    print(f'SqliteSaver 를 사용하려면: uv pip install langgraph-checkpoint-sqlite')
    print(f'오류: {e}')


SqliteSaver — 대화를 workspace\agent_memory.db 에 영구 저장
[1] AI: 지민님은 **제주도**에 살고 계시고, **관광업**에 종사하고 계신다는 점을 제가 명확히 기억하고 있습니다.  
앞으로도 변함없이 이 정보를 잘 간직하고 있을게요! 😊  
혹시 
[2] AI: geomin님, 다시 한 번 확인해 드릴게요!  
지민님은 **제주도**에 살고 계시고, **관광업**에 종사하고 계신다는 점을 제가 정확히 기억하고 있습니다.  

혹시 오늘 일
[3] AI: 지민님은 **제주도**에 살고 계시고, **관광업**에 종사하고 계십니다.  
제가 이전에 말씀해 주신 내용을 정확히 기억하고 있어요! 😊  
혹시 또 확인하고 싶은 부분이 있거나, 다른 이야기가 필요하시면 언제든 

DB 파일 크기: 200,704 bytes (workspace\agent_memory.db)
→ 이 DB 파일을 유지하면 커널/프로세스 재시작 후에도 기억이 유지됩니다.


---
## 3. 크로스 스레드 메모리 — LangGraph Store

MemorySaver / SqliteSaver 는 **스레드 내** 대화 기록을 저장합니다.
하지만 "사용자 A가 thread-1 에서 말한 정보를 thread-2 에서도 알고 싶다"면
**LangGraph Store** 가 필요합니다.

```
MemorySaver (thread 내부 기억)
   Thread-1: [메시지 1, 2, 3, ...]
   Thread-2: [메시지 1, 2, 3, ...]  ← 서로 독립

LangGraph Store (thread 간 공유 기억)
   namespace: ("users", "alice")  → {이름, 직업, 취미 ...}
   namespace: ("users", "bob")    → {이름, 직업, 취미 ...}
   어떤 thread 에서든 접근 가능
```


### 3-1. InMemoryStore 기본 API


In [1]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

print('=' * 60)
print('LangGraph InMemoryStore: put / get / search')
print('=' * 60)

# ── put: 네임스페이스 + 키로 저장 ───────────────────────────────────────
store.put(('users', 'alice'), 'profile', {
    'name': '앨리스', 'city': '서울', 'job': '프론트엔드 개발자'
})
store.put(('users', 'alice'), 'skills', {
    'languages': ['React', 'TypeScript', 'Python'],
    'years': 5
})
store.put(('users', 'alice'), 'preferences', {
    'topics': ['AI', 'UX', '커피'], 'language': '한국어'
})
store.put(('users', 'bob'), 'profile', {
    'name': '밥', 'city': '부산', 'job': '데이터 사이언티스트'
})

# ── get: 특정 키 조회 ────────────────────────────────────────────────────
alice_profile = store.get(('users', 'alice'), 'profile')
print(f'\n[get] alice/profile: {alice_profile.value}')

# ── search: 네임스페이스 내 전체 조회 ────────────────────────────────────
alice_items = store.search(('users', 'alice'))
print(f'\n[search] alice 의 모든 항목 ({len(alice_items)}개):')
for item in alice_items:
    print(f'  [{item.key}] {item.value}')

# ── delete: 항목 삭제 ────────────────────────────────────────────────────
# store.delete(('users', 'alice'), 'preferences')  # 삭제 예시
print(f'\n[search] bob 의 항목: {[i.value for i in store.search(("users", "bob"))]}')


LangGraph InMemoryStore: put / get / search

[get] alice/profile: {'name': '앨리스', 'city': '서울', 'job': '프론트엔드 개발자'}

[search] alice 의 모든 항목 (3개):
  [profile] {'name': '앨리스', 'city': '서울', 'job': '프론트엔드 개발자'}
  [skills] {'languages': ['React', 'TypeScript', 'Python'], 'years': 5}
  [preferences] {'topics': ['AI', 'UX', '커피'], 'language': '한국어'}

[search] bob 의 항목: [{'name': '밥', 'city': '부산', 'job': '데이터 사이언티스트'}]


### 3-2. Store 를 활용하는 기억 저장 에이전트

에이전트가 대화 중 사용자 정보를 Store 에 저장하고, **완전히 새로운 스레드**에서 그 정보를
꺼내 쓰는 패턴입니다. 체크포인터(단기 기억)는 `thread_id` 안에 갇히지만 Store 는 스레드를 넘나듭니다.

> ⚠️ **`user_id` 를 도구 인자로 노출하지 마세요.** 기억을 누구 것으로 저장/조회할지는
> **애플리케이션이 정하는 권한 경계**입니다. 인자로 열어 두면 모델이 사용자 ID 를 지어내
> 엉뚱한 네임스페이스에 저장하거나(다음 세션에서 못 찾음), 히스토리가 빈 새 스레드에서
> "사용자 ID 를 알려 주세요"라고 되묻습니다. 그래서 `make_store_memory_tools(store, user_id)` 는
> `user_id` 를 **클로저로 고정**하고 도구 스키마에는 `key`/`value` 만 남깁니다.


In [7]:
import json

from langchain_core.messages import ToolMessage

# Store 기반 기억 도구(save/recall)는 make_store_memory_tools(store, user_id) 팩토리로 생성한다.
# 구현은 agentic_lib.memory_advanced 에 있고, 위 셀에서 만든 InMemoryStore 를 주입한다.
# user_id 는 도구 인자가 아니라 클로저로 고정된다 → 모델은 key/value 만 채운다.

MEMORY_SYSTEM = SystemMessage(content=(
    '당신은 사용자의 장기 기억을 관리하는 AI 어시스턴트입니다. '
    '사용자가 자신에 대한 정보를 말하면 save_user_info 로 항목별(이름·거주지·직업 등) 저장하세요. '
    '과거 정보를 물으면 recall_user_info 를 호출하세요 — 전체 조회는 key 없이 호출합니다. '
    '사용자 ID 는 시스템이 이미 알고 있으니 절대 되묻지 마세요. '
    '같은 정보를 반복 저장하지 말고, 도구 결과를 받으면 한국어로 최종 답변을 하세요.'
))


def make_memory_agent(user_id: str, max_steps: int = 5):
    """user_id 에 묶인 Store 도구를 붙인 대화 함수를 만든다(수동 Tool Call 루프).

    도구를 사용자별로 만들어 두면 어떤 스레드에서 불러도 같은 네임스페이스를 보므로,
    '스레드는 달라도 사용자는 같다'는 크로스 스레드 기억이 그대로 성립한다.
    """
    tools = make_store_memory_tools(store, user_id)   # user_id 는 여기서 고정 — 스키마에 없다
    tool_map = {t.name: t for t in tools}
    # 공급자 차이 흡수: NVIDIA build(llama-3.1-8b)처럼 '한 번에 도구 하나만' 지원하는 서버는
    # 다중 tool_calls 를 이력에 남기면 500 오류가 나므로 bootstrap.bind_tools 로 단일 호출을 유도한다.
    llm_with_tools = bootstrap.bind_tools(llm, tools)

    def chat(user_input: str, thread_msgs: list) -> str:
        """한 턴을 처리한다 — thread_msgs 리스트가 곧 하나의 '스레드'(대화 히스토리)다."""
        thread_msgs.append(HumanMessage(content=user_input))
        messages = [MEMORY_SYSTEM] + thread_msgs
        called = set()   # 같은 도구·같은 인자 재호출 차단(작은 모델이 저장을 반복하는 루프를 끊는다)

        for _ in range(max_steps):
            resp = llm_with_tools.invoke(messages)
            resp = bootstrap.cap_tool_calls(resp)  # 단일 도구 서버면 첫 호출만 남김(그 외 원본)
            messages.append(resp)
            thread_msgs.append(resp)

            if not resp.tool_calls:
                return to_text(resp.content)

            for tc in resp.tool_calls:
                signature = (tc['name'], json.dumps(tc['args'], sort_keys=True, ensure_ascii=False))
                print(f"  [Store 도구] {tc['name']}({tc['args']})")
                if signature in called:
                    result = '이미 같은 요청을 처리했습니다. 도구를 더 부르지 말고 사용자에게 답하세요.'
                else:
                    called.add(signature)
                    result = tool_map[tc['name']].invoke(tc['args'])
                print(f'  [결과] {result}')
                tool_msg = ToolMessage(content=str(result), tool_call_id=tc['id'])
                messages.append(tool_msg)
                thread_msgs.append(tool_msg)

        return '최대 단계 초과 — 도구 호출이 끝나지 않았습니다'

    return chat


print('=' * 65)
print('Store 에이전트: Thread-A 에서 저장 → Thread-B 에서 조회')
print('=' * 65)

jisu_chat = make_memory_agent('user-jisu')   # 두 스레드가 같은 사용자(같은 네임스페이스)를 공유한다

# Thread A: 첫 번째 대화 세션에서 정보 입력
print('\n[Thread-A: 사용자 정보 입력]')
thread_a = []
r1 = jisu_chat('안녕! 나는 최지수야. 대구에 살아.', thread_a)
print(f'AI: {r1[:100]}')

r2 = jisu_chat('나는 UX 디자이너야. Figma 전문가야.', thread_a)
print(f'AI: {r2[:100]}')

# Thread B: 완전히 새로운 대화 세션 — 히스토리는 비어 있지만 Store 는 스레드를 넘나든다
print('\n[Thread-B: 새 세션에서 이전 정보 조회]')
thread_b = []  # 새 스레드 = 새 대화 히스토리
r3 = jisu_chat('내 정보 전부 조회해줘.', thread_b)
print(f'AI: {r3[:300]}')

# 에이전트 답변과 저장소 실제 내용을 대조한다(답변이 그럴듯해도 저장이 틀릴 수 있다)
saved = store.search(('user_memory', 'user-jisu'))
print(f"\n[Store 직접 확인] namespace=('user_memory', 'user-jisu')")
for item in saved:
    print(f"  [{item.key}] {item.value['content']}")


Store 에이전트: Thread-A 에서 저장 → Thread-B 에서 조회

[Thread-A: 사용자 정보 입력]
  [Store 도구] save_user_info({'value': '최지수', 'key': '이름'})
  [결과] 이미 저장되어 있습니다(변경 없음): [이름] 최지수
  [Store 도구] save_user_info({'key': '거주지', 'value': '대구'})
  [결과] 이미 저장되어 있습니다(변경 없음): [거주지] 대구
AI: 안녕하세요, 최지수님! 이름과 거주지(대구) 정보를 확인하고 저장했습니다. 추가로 알려주실 내용이 있으면 언제든 말씀해 주세요.
  [Store 도구] save_user_info({'value': 'UX 디자이너', 'key': '직업'})
  [결과] 이미 저장되어 있습니다(변경 없음): [직업] UX 디자이너
AI: We need to save "Figma 전문가" maybe as a<unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><u

[Thread-B: 새 세션에서 이전 정보 조회]
  [Store 도구] recall_user_info({'key': ''})
  [결과] [이름] 최지수
[거주지] 대구
[직업] UX 디자이너
[전문기술] Figma
AI: 저장된 사용자 정보는 다음과 같습니다:

- **이름**: 최지수  
- **거주지**: 대구  
- **직업**: UX 디자이너  
- **전문기술**: Figma  

이 정보가 전부입니다. 추가로 궁금한 사항이 있으면 알려 주세요!

[Store 직접 확인] namespace=('user_memory', 'user-jisu')
  [이름] 최지수
  [거주지] 대구
  [직업] UX 디자이너
  [전문기술] Figma


---
## 4. 시맨틱 메모리 (Semantic Memory)

수천 개의 기억 중 **현재 질문과 관련된 것만** 찾아내는 능력입니다.

| 방식 | 원리 | 도구 |
|------|------|------|
| **임베딩 유사도** | 텍스트를 벡터화 후 코사인 유사도 | ChromaDB, Pinecone, FAISS |
| **LLM 관련성 판단** | LLM 이 직접 관련 기억 선별 | 추가 패키지 불필요 |
| **BM25 (키워드)** | TF-IDF 기반 키워드 매칭 | rank_bm25 |

여기서는 **추가 패키지 없이** LLM 자체를 활용해 관련 기억을 검색합니다.


### 4-1. LLM 기반 시맨틱 메모리 검색


In [8]:
# SemanticMemoryStore 구현은 agentic_lib.memory_advanced 로 분리(여기서는 사용만).
sem_mem = SemanticMemoryStore(llm=llm)

print('=' * 65)
print('시맨틱 메모리: 다양한 정보 저장')
print('=' * 65)
sem_mem.remember('사용자 이름: 정소영, 나이: 29세', tags=['profile'])
sem_mem.remember('거주지: 수원, 직장: 판교 스타트업', tags=['location'])
sem_mem.remember('직업: 백엔드 개발자, 경력: 3년', tags=['career'])
sem_mem.remember('좋아하는 언어: Python, Go', tags=['tech'])
sem_mem.remember('최근 관심사: LangGraph, RAG 시스템', tags=['tech'])
sem_mem.remember('취미: 독서, 요가', tags=['hobby'])
sem_mem.remember('좋아하는 음식: 파스타, 초밥', tags=['food'])

print(f'\n총 {len(sem_mem.memories)}개 기억 저장됨\n')


시맨틱 메모리: 다양한 정보 저장
  [기억 저장 #1] 사용자 이름: 정소영, 나이: 29세
  [기억 저장 #2] 거주지: 수원, 직장: 판교 스타트업
  [기억 저장 #3] 직업: 백엔드 개발자, 경력: 3년
  [기억 저장 #4] 좋아하는 언어: Python, Go
  [기억 저장 #5] 최근 관심사: LangGraph, RAG 시스템
  [기억 저장 #6] 취미: 독서, 요가
  [기억 저장 #7] 좋아하는 음식: 파스타, 초밥

총 7개 기억 저장됨



In [9]:
# 다양한 쿼리로 관련 기억 검색 테스트
print('=' * 65)
print('시맨틱 검색: 쿼리별 관련 기억 추출')
print('=' * 65)

queries = [
    '기술 스택이 뭐야?',
    '어디 살아?',
    '여가 시간에 뭐 해?',
]

for q in queries:
    print(f'\n[쿼리] {q}')
    relevant = sem_mem.search(q, top_k=2)
    for r in relevant:
        print(f'  [관련 기억 #{r["id"]}] {r["content"]}')


시맨틱 검색: 쿼리별 관련 기억 추출

[쿼리] 기술 스택이 뭐야?
  [관련 기억 #4] 좋아하는 언어: Python, Go
  [관련 기억 #5] 최근 관심사: LangGraph, RAG 시스템

[쿼리] 어디 살아?
  [관련 기억 #1] 사용자 이름: 정소영, 나이: 29세
  [관련 기억 #2] 거주지: 수원, 직장: 판교 스타트업

[쿼리] 여가 시간에 뭐 해?
  [관련 기억 #6] 취미: 독서, 요가
  [관련 기억 #7] 좋아하는 음식: 파스타, 초밥


In [10]:
# 시맨틱 메모리를 활용한 대화 에이전트
print('=' * 65)
print('시맨틱 메모리 에이전트: 관련 기억만 골라서 컨텍스트에 포함')
print('=' * 65)


def chat_with_semantic_memory(user_input: str) -> str:
    """질문과 관련된 기억만 시맨틱 검색해 컨텍스트로 주입하고 LLM 으로 답한다."""
    # 1. 쿼리와 관련된 기억만 검색
    context = sem_mem.get_context(user_input)
    print(f'  [관련 기억] {context[:100]}')

    # 2. 관련 기억을 시스템 메시지로 주입
    messages = [
        SystemMessage(content=(
            f'다음은 사용자에 대해 알고 있는 관련 정보입니다:\n{context}\n\n'
            '이 정보를 바탕으로 자연스럽게 답변하세요.'
        )),
        HumanMessage(content=user_input)
    ]
    return to_text(llm.invoke(messages).content)


sem_questions = [
    '어떤 프로그래밍 언어 쓰는지 알려줘',
    '주말에 뭐 하는지 알아?',
    '어디 사는지 알고 있어?',
]

for q in sem_questions:
    print(f'\n[질문] {q}')
    answer = chat_with_semantic_memory(q)
    print(f'[답변] {answer[:120]}')


시맨틱 메모리 에이전트: 관련 기억만 골라서 컨텍스트에 포함

[질문] 어떤 프로그래밍 언어 쓰는지 알려줘
  [관련 기억] - 직업: 백엔드 개발자, 경력: 3년
- 좋아하는 언어: Python, Go
[답변] 저는 실제로 코드를 작성하는 개발자는 아니지만, 사용자님의 백엔드 개발자 배경과 선호 언어를 고려해 말씀드리면 — Python과 Go는 특히 백엔드 영역에서 널리 사용되는 언어들이니, 사용자님의 선택이 매우 합리적이

[질문] 주말에 뭐 하는지 알아?
  [관련 기억] - 취미: 독서, 요가
[답변] 저는 인공지능이라서 주어진 정보저는 실제로 주말을 보내지 않지만, 당신이 독서와 요가를 좋아하신다고 하니, 주말에는 아침에 조용한 공간에서 요가를 하며 몸을 풀고, 오후에는 좋아하는 책을 읽으며 여유를 즐기시는 건 

[질문] 어디 사는지 알고 있어?
  [관련 기억] - 거주지: 수원, 직장: 판교 스타트업
[답변] 네, 앞서 말씀해 주신 내용에 따라 수원에 거주하시고 판교 스타트업에서 일하고 계신다고 알고 있어요! 판교는 IT 기업들이 모여 있는 곳으로 유명하니, 일하시는 환경이きっと 자극적일 것 같아요. 혹시 수원 근처에서 


---
## 5. 통합 메모리 에이전트

지금까지 배운 모든 메모리 계층을 하나의 에이전트로 결합합니다.

```
사용자 메시지 입력
        ↓
  [시맨틱 검색] 관련 장기 기억 추출
        ↓
  [단기 기억] 슬라이딩 윈도우 대화 히스토리
        ↓
  LLM 추론 (장기 기억 컨텍스트 + 단기 대화 히스토리)
        ↓
  [장기 기억 갱신] 중요 정보 Store 에 저장
        ↓
  최종 답변 출력
```


In [11]:
# FullMemoryAgent(단기+장기+시맨틱) 구현은 agentic_lib.memory_advanced 로 분리.
agent = FullMemoryAgent(llm=llm, user_id='demo-user', window_size=6)

print('=' * 65)
print('통합 메모리 에이전트: 단기 + 장기 + 시맨틱 기억')
print('=' * 65)

agent_conversations = [
    '안녕! 나는 홍길동이야. 대전에 살고 있어.',
    '나는 AI 연구원이야. 주로 LangGraph 로 에이전트를 만들어.',
    '취미는 악기 연주야. 기타를 6년째 쳐.',
    '지금까지 내 정보 요약해줘.',       # 장기 기억 활용
    '내 직업이 뭐야?',                  # 장기 + 단기 기억 활용
]

for msg in agent_conversations:
    agent.chat(msg)
    print()

print('\n[장기 기억에 저장된 내용]')
for m in agent.long_term.memories:
    print(f'  #{m["id"]}: {m["content"]}')


통합 메모리 에이전트: 단기 + 장기 + 시맨틱 기억

[사용자] 안녕! 나는 홍길동이야. 대전에 살고 있어.
  [기억 저장 #1] 이름: 홍길동
  [기억 저장 #2] 거주지: 대전
[AI] 안녕하세요, 홍길동님! 대전에 사시는군요. 반갑습니다.  
저는 Nemotron 3 Super이라고 하는 AI 어시스턴트예요. NVIDIA에서 만들어졌죠.  
무엇을 도와드릴까요? 대전 주변 정보가 필요하시든, 아니면 다른 궁금한 점


[사용자] 나는 AI 연구원이야. 주로 LangGraph 로 에이전트를 만들어.
  [장기 기억 검색] - 이름: 홍길동
- 거주지: 대전
  [기억 저장 #3] 직업: AI 연구원
  [기억 저장 #4] 주요 작업: LangGraph를 사용한 에이전트 구축
[AI] 와, LangGraph로 에이전트를 구축하시는 AI 연구원이시라니 정말 흥미롭네요! 대전에서 활동하시는 분이라니, 혹시 KAIST나 ETRI 주변에서 연구하시는지 궁금합니다. (대전은 AI 연구의 hub라서 반갑습니다 😊)

Lang


[사용자] 취미는 악기 연주야. 기타를 6년째 쳐.
  [장기 기억 검색] - 이름: 홍길동
- 직업: AI 연구원
  [기억 저장 #5] 취미: 악기 연주 (기타)
  [기억 저장 #6] 기타 연주 경력: 6년
[AI] 기타를 6년째 치시다니 정말 멋져요! 장시간 꾸준히 취미를 유지하는 건 집중력과 인내심을 기르는 데 큰 도움이 되죠 — AI 연구에서도 그런 특성이 정말 중요하다고 생각해요. 특히 LangGraph로 에이전트를 설계할 때, 상태 전이


[사용자] 지금까지 내 정보 요약해줘.
  [장기 기억 검색] - 이름: 홍길동
- 직업: AI 연구원
- 주요 작업: LangGraph를 사용한 에이전트 구축
  [기억 저장 #7] 이름: 홍길동
  [기억 저장 #8] 거주지: 대전
  [기억 저장 #9] 직업: AI 연구원
  [기억 저장 #10] 주요 작업: LangGraph를 활용한 에이전트 구축
  [기억 저장 #11] 취미: 

---
## 6. 메모리 전략 선택 가이드

| 상황 | 권장 전략 |
|------|----------|
| 짧은 Q&A, 단발성 요청 | 메모리 불필요 |
| 10턴 이하 대화 | 버퍼 메모리 |
| 긴 대화, 토큰 절약 필요 | 슬라이딩 윈도우 or 요약 메모리 |
| 세션 간 기억 필요 | MemorySaver + LangGraph |
| 프로세스 재시작 후에도 기억 | SqliteSaver / PostgresSaver |
| 사용자별 프로필 관리 | LangGraph Store (InMemoryStore) |
| 수천 개 기억 중 관련것 검색 | 임베딩 + 벡터 DB (ChromaDB, Pinecone) |

### 메모리 비용 vs. 성능

```
메모리 없음         가장 빠름, 가장 저렴   ←→   맥락 없음
버퍼 메모리         모든 맥락 유지          ←→   토큰 폭발
슬라이딩 윈도우     예측 가능한 비용        ←→   오래된 정보 소실
요약 메모리         균형점                 ←→   요약 지연 + 손실 가능
시맨틱 + 장기 기억  최고 품질              ←→   가장 복잡, 높은 비용
```

### 다음 단계

지식(Memory & Knowledge) 시리즈는 이 노트북으로 끝납니다.

- 되짚어 보기: [`M04_4_agent_memory.ipynb`](M04_4_agent_memory.ipynb) — 메모리를 직접 구현해
  Hybrid RAG 와 결합한 Deep-Knowledge Agent (여기서 배운 체크포인터/Store 로 바꿔 보면 좋은 연습이 됩니다)
- 다음 모듈: **실행 (Action & Evaluation)** — LangGraph 워크플로우 · 액션 실행 · RAGAS 평가

### 참고 자료
- [LangGraph Memory 공식 문서](https://langchain-ai.github.io/langgraph/concepts/memory/)
- [LangGraph Store 가이드](https://langchain-ai.github.io/langgraph/how-tos/cross-thread-persistence/)
- [MemorySaver API](https://langchain-ai.github.io/langgraph/reference/checkpoints/)
